In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
def getParameter(dataFrame, file_names, timeScale, export=False, regionalize=False, logarithmic=False):

    dataFrame = subBlank(dataFrame)
    data = {}
    
    for label in dataFrame.__iter__():
        if np.nanmax(dataFrame[label]) - np.nanmin(dataFrame[label]) < 0.01:
            maxOD, maxRate, deathRate, lagTime, growthPhase = 0, 0, 0, 0, 0
            maxODPoint, maxODCheck, maxRatePoint, deathRatePoint, boundaryPoint1, boundaryPoint2 = 0, 0, 0, 0, 0, 0
        else:
            try:
                maxODIndex, maxOD, maxODCheck = getMaxOD(dataFrame[label])
            except:
                maxODIndex, maxOD, maxODCheck = np.nan, np.nan, 1
            try:
                maxODPoint = dataFrame.index[maxODIndex]
            except:
                maxODPoint = np.nan
            try:
                maxRatePoint, maxRate = getMaxGradient(dataFrame[label],timeScale, regionalize)
            except:
                maxRatePoint, maxRate = np.nan, np.nan
            try:
                deathRatePoint, deathRate = getDeathRate(dataFrame[label],timeScale, regionalize, logarithmic)
            except:
                deathRatePoint, deathRate = np.nan, np.nan
            try:
                lagTimeIndex = getLagPhase(dataFrame[label])
            except:
                lagTimeIndex = np.nan
            try:
                lagTime = dataFrame.index.values[lagTimeIndex]
            except:
                lagTime = np.nan
            try:
                growthPhase = dataFrame.index.values[maxODIndex - lagTimeIndex]
            except:
                growthPhase = np.nan
            try:
                boundaryPoint1, boundaryPoint2 = dataFrame.index.values[getBoundary(dataFrame[label],timeScale)]
            except:
                boundaryPoint1, boundaryPoint2 = np.nan, np.nan
        data[label] = [ maxOD, maxRate, deathRate, lagTime, growthPhase, maxODPoint, maxODCheck, maxRatePoint, deathRatePoint, boundaryPoint1, boundaryPoint2]
    
    parameter = pd.DataFrame(data,
                             columns=[label for label in dataFrame.__iter__()],
                             index=["MaxOD","MaxRate","DeathRate","LagTime","GrowthPhase","MaxODPoint","MaxODCheck", "MaxRatePoint","DeathRatePoint","BoundaryPoint1","BoundaryPoint2"])
    
    parameter = parameter.T
    
    if export:
        folder_path = os.getcwd() + "/../param"
        if logarithmic:
            file_name = "parameter_" + file_names +"_log.xlsx"
        else:
            file_name = "parameter_" + file_names +".xlsx"
        file_path = folder_path + "/" + file_name
        if(not os.path.isdir(folder_path)):
            print("mkdir: " + folder_path)
            os.mkdir(folder_path)
        parameter.to_excel(file_path)
        print("Params are exported to [" + file_path + "]")
    return parameter

def subBlank(dataFrame):
    df = dataFrame.copy()
    
    for column_name in df.columns:
        series = df[column_name].astype(float)
        blank = series.min() - 0.0001
        
        if blank > 0.09:
            blank = 0.09
        series = series - blank
        series = series.where(series > 0, np.nan).bfill()
        
        if series.isna().all():
            series = series.fillna(0.001)
        df[column_name] = series       
    return df

def getMaxOD(series):
    series_ave = series.rolling(3, center=True, min_periods=3).mean()
    series_check = series_ave.dropna()
    check_num = 0
    
    if len(series_check) == 0:
        return (np.nan, np.nan, 1)
    index = int(np.nanargmax(series_ave.values))
    
    if series_ave.index[index] == series_check.index[-1]:
        check_num = 3
    return (index, series_ave.iloc[index], check_num)

def getMaxGradient(series,timeScale,regionalize):
    ser_ori = series.copy()
    min_num = 5
    threshold = 0.0005


    if regionalize:
        t_l, t_s = getBoundary(series,timeScale)
        while True:
            if series.values[t_l -1]== series.min() or series.values[t_l]== series.min() :
                t_l = t_l + 1
            else:
                break
        series = series[t_l:t_s]
        
                
    for i in range(series.size):
        if series.iloc[i] > threshold:
            series = series[i:]
            break

    log_values = np.log(series.values)
    gradients = np.gradient(log_values, timeScale)

    time = list(series.index)
    
    Q1 = stats.scoreatpercentile(gradients, 25)
    Q3 = stats.scoreatpercentile(gradients, 75)
    h_width = Q3 - Q1
    abline = Q3 + h_width*1.5
    while True:
        index = np.argmax(gradients)
        if abline < gradients[index]:
            gradients = np.delete(gradients, index)
            time.pop(index)
        else:
            break

    gradients = pd.Series(gradients)
    gradients = gradients.rolling(3, center=True, min_periods=1).mean()
    index = np.argmax(gradients)

    return (series.index[index], gradients[index])

def getDeathRate(series,timeScale, regionalize, logarithmic):
    
    if regionalize:
        t_o = getMaxOD(series)[0]
        series = series[t_o:]

    if logarithmic:
        log_values = np.log(series.values)
        gradients = np.gradient(log_values, timeScale)
    else:
        gradients = np.gradient(series.values, timeScale)
    time = list(series.index)
    Q1 = stats.scoreatpercentile(gradients, 25)
    Q3 = stats.scoreatpercentile(gradients, 75)
    h_width = Q3 - Q1
    abline = Q3 + h_width*1.5
    while True:
        index = np.argmax(gradients)
        if abline < gradients[index]:
            gradients = np.delete(gradients, index)
            time.pop(index)
        else:
            break

    gradients = pd.Series(gradients)
    gradients = gradients.rolling(3, center=True, min_periods=1).mean()
    index = np.argmin(gradients)

    return (series.index[index], gradients[index])

def getLagPhase(series):
    p = np.nan
    for i in range(4,len(series)):
        if((series.iloc[i]>series.iloc[i-1])&(series.iloc[i-1]>series.iloc[i-2])&(series.iloc[i-2]>series.iloc[i-3])&(series.iloc[i-3]>series.iloc[i-4])):
            p = i
            break
    return  p

def getBoundary(series,timeScale):
    min_num = 5
    boundary = [-1]    
    series= series.dropna(how="all")
    minOD = series.min()
    minODTime = series.idxmin()
    for value in series.index[series.index < minODTime]:
        series[value] = minOD

    
    boundary[0] = getLagPhase(series)
    
     
    b1 = pd.Series(np.log(series.values[boundary[0]:]))
    gradients = np.gradient(b1.rolling(5, center=True, min_periods=1).mean(), timeScale)
    boundary = boundary + [-1]
    min_unfitness = float("inf")

    for t_ls in range(min_num, len(gradients)-min_num):
        unfitness = (np.var(gradients[:t_ls]) +
                     np.var(gradients[t_ls:]))
        if(unfitness < min_unfitness):
            boundary[1] = boundary[0] + t_ls
            min_unfitness = unfitness

    y = np.log(series.values[:boundary[1]] + 0.08)        
    min_unfitness = float("inf")
    for t_ll in range(min_num, len(y)-min_num):
        A = np.array([series.index[:t_ll],np.ones(len(series.index[:t_ll]))],dtype=float).T
        unfitness = np.linalg.lstsq(A, y[:t_ll],rcond=-1)[1] / t_ll
        A = np.array([series.index[t_ll:boundary[1]],np.ones(len(series.index[t_ll:boundary[1]]))],dtype=float).T
        unfitness = unfitness + np.linalg.lstsq(A, y[t_ll:],rcond=-1)[1] / (boundary[1] - t_ll)

        if(unfitness < min_unfitness):
            boundary[0] = t_ll
            min_unfitness = unfitness

    return boundary

def getMergedParamData(param_file1, param_file2, cond_file):
    print("Reading files...")
    
    df_param1 = pd.read_excel(param_file1, index_col=0)
    df_param2 = pd.read_excel(param_file2, index_col=0)
    df_param = pd.concat([df_param1, df_param2])
    print(f"Merged param data shape: {df_param.shape}")
    
    df_cond = pd.read_excel(cond_file, index_col=0)
    
    paramData = df_param.join(df_cond["Condition_ID"].rename("condition"), how="left")
    
    paramData = paramData.dropna(subset=["condition"])
    cols = paramData.columns.tolist()
    cols.insert(0, cols.pop(cols.index("condition")))
    paramData = paramData[cols]    
    parameters = ["MaxOD", "MaxRate", "DeathRate", "LagTime", "GrowthPhase"]
    conditions = sorted(paramData["condition"].unique().tolist())
    
    print(f"Data preparation complete. Total wells: {len(paramData)}, Total conditions: {len(conditions)}")
    
    return paramData, parameters, conditions

def aggregateParam(paramData, parameters, conditions, file, alpha):
    paramData_new = paramData.copy()
    aggData = pd.DataFrame(index=conditions)
    
    for param in parameters:
        print("Processing: " + param)
        info = f"{param}_info"
        dic_info_all = {}
        dic_agg_all = {}
        
        for cond in conditions:
            df = paramData_new[paramData_new["condition"] == cond][[param, "MaxODCheck"]].copy()
            dic_info, dic_agg = SmirnovGrubbs(df, alpha)
            dic_info_all.update(dic_info)
            dic_agg_all[cond] = dic_agg
            
        df_info = pd.DataFrame.from_dict(dic_info_all, orient="index", columns=[info])
        df_agg = pd.DataFrame.from_dict(dic_agg_all, orient="index")
        
        paramData_new = pd.concat([paramData_new, df_info], axis=1, sort=False)
        aggData = pd.concat([aggData, df_agg], axis=1, sort=False)
        
    print("Processing is complete")
        
    folder_path = os.path.abspath(os.path.join(os.getcwd(), "..", "param", "agg"))
    os.makedirs(folder_path, exist_ok=True)
    
    file_path_agg = os.path.join(folder_path, f"{file}_agg{alpha}.xlsx")
        
    with pd.ExcelWriter(file_path_agg, engine='openpyxl') as writer:
        paramData_new.to_excel(writer, sheet_name='parameter')
        aggData.to_excel(writer, sheet_name='aggregate')

def SmirnovGrubbs(df, alpha):
    df_n = len(df)
    df_ave = np.nan 
    df_sd = np.nan 
    df_val = 0 
    df_info = 2 
    
    df["info"] = 2
    param = df.columns[0]
    
    if param in ["MaxOD", "GrowthPhase"]:
        ex = df[df["MaxODCheck"] != 0].index
        df.loc[ex, "info"] = df.loc[ex, "MaxODCheck"]
        
    try:
        while True:
            x = df.dropna()
            x = x[x["info"] == 2]
            x = x.iloc[:, 0]
            n = len(x)
            
            if n <= 2:
                if n > 0:
                    df.loc[x.index, "info"] = 0
                    df_ave = np.mean(x)
                    df_sd = np.std(x)
                    df_val = n
                break
                
            t = stats.t.isf(q=(alpha/n)/2, df=n-2)
            tau = (n-1) * t / np.sqrt((n * (n-2) + n * t**2))
            
            min_value = x.idxmin()
            max_value = x.idxmax()
            mean = np.mean(x)
            sd = np.std(x)
            
            if sd == 0.0:
                df.loc[x.index, "info"] = 0
                df_ave, df_sd, df_val = mean, sd, n
                break
                
            min_dist = abs(x[min_value] - mean)
            max_dist = abs(x[max_value] - mean)
            
            out = min_value if min_dist > max_dist else max_value
            dist = abs(x[out] - mean) / sd
            
            if tau < dist:
                df.at[out, "info"] = 1
            else:
                df.loc[x.index, "info"] = 0
                df_ave, df_sd, df_val = mean, sd, n
                break 
                
        df_bool = (df["info"] == 0)
        
        if df_bool.sum() > 3:
            df_info = 0
        elif (df[param] == 0).all():
            df_info = 0
        else:
            df_info = 1
            
        dic_info = df["info"].to_dict()
        dic_agg = {f"{param}_ave": df_ave, f"{param}_sd": df_sd, f"{param}_valid": df_val, f"{param}_info": df_info}
        
        return dic_info, dic_agg
        
    except Exception as e:
        dic_info = df["info"].to_dict()
        dic_agg = {f"{param}_ave": df_ave, f"{param}_sd": df_sd, f"{param}_valid": df_val, f"{param}_info": df_info}
        return dic_info, dic_agg

In [ ]:
file_names = ["Curves_Part I", "Curves_Part II"]
for file_name in file_names:
    data = pd.read_excel(file_name, index_col=0)
    timeScale = 0.5
    getParameter(data, file_name,timeScale, export=True, regionalize=True, logarithmic= False)

mkdir: c:\Users\taro\OneDrive - 筑波大学\lab\data\data_paper\scr/../param
Params are exported to [c:\Users\taro\OneDrive - 筑波大学\lab\data\data_paper\scr/../param/parameter_Curves_Part I.xlsx.xlsx]
Params are exported to [c:\Users\taro\OneDrive - 筑波大学\lab\data\data_paper\scr/../param/parameter_Curves_Part II.xlsx.xlsx]


In [5]:
param_file1 = os.path.join("../param/parameter_Curves_Part I.xlsx")
param_file2 = os.path.join("../param/parameter_Curves_Part II.xlsx")
cond_file = "Summary_260806.xlsx"

paramData, parameters, conditions = getMergedParamData(param_file1, param_file2, cond_file)

Reading files...
Merged param data shape: (11028, 11)
Data preparation complete. Total wells: 11028, Total conditions: 1481


In [6]:
aggregateParam(paramData, parameters, conditions, file="All_Merged_Data", alpha=0.01)

Processing: MaxOD
Processing: MaxRate
Processing: DeathRate
Processing: LagTime
Processing: GrowthPhase
Processing is complete
